## Step 1: Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from pathlib import Path
from pdf2image import convert_from_path
import hashlib
from datetime import datetime

print("Libraries imported successfully")

## Step 2: Configure Folder and Directory Structure

In [ ]:
# Configure the folder to search for PDF/PPTX files
# Change this to your target folder
search_folder = Path(r"C:\structure\temp\slides")  # Modify this path as needed

# Create the output directory structure
base_dir = Path("./")
images_dir = base_dir / "images"
images_dir.mkdir(parents=True, exist_ok=True)

# Database file path
database_path = base_dir / "data.csv"

print(f"Search folder: {search_folder}")
print(f"Images output directory: {images_dir}")
print(f"Database path: {database_path}")

## Step 3: Find PDF/PPTX Pairs

In [ ]:
def find_pdf_pptx_pairs(search_folder):
    """
    Recursively search for PDF files that have matching PPTX files.
    Returns a list of (pdf_path, pptx_name) tuples.
    """
    pairs = []
    
    # Find all PDF and PPTX files
    pdf_files = {}
    pptx_files = set()
    
    for root, dirs, files in os.walk(search_folder):
        for file in files:
            file_path = Path(root) / file
            file_stem = file_path.stem
            
            if file.lower().endswith('.pdf'):
                pdf_files[file_stem] = file_path
            elif file.lower().endswith(('.pptx', '.ppt')):
                pptx_files.add(file_stem)
    
    # Find matching pairs
    for stem in pdf_files:
        if stem in pptx_files:
            pairs.append((pdf_files[stem], stem))
    
    return pairs

# Find all PDF/PPTX pairs
pdf_pptx_pairs = find_pdf_pptx_pairs(search_folder)
print(f"Found {len(pdf_pptx_pairs)} PDF/PPTX pairs")
for pdf_path, pptx_name in pdf_pptx_pairs[:5]:  # Show first 5
    print(f"  {pdf_path.name} <- {pptx_name}")

## Step 4: Load Existing Database (if exists)

In [ ]:
def load_database(database_path):
    """
    Load the existing database or create a new empty one.
    """
    if database_path.exists():
        df = pd.read_csv(database_path)
        # Convert embedding strings back to numpy arrays
        if 'embedding' in df.columns and len(df) > 0:
            df['embedding'] = df['embedding'].apply(
                lambda x: np.array([float(v) for v in x.split(',')]) if isinstance(x, str) else x
            )
        print(f"Loaded existing database with {len(df)} entries")
        return df
    else:
        print("No existing database found. Creating new one.")
        return pd.DataFrame(columns=['filename', 'source_pptx', 'pdf_file', 'slide_number', 'pdf_hash', 'embedding', 'x', 'y', 'z'])

def is_pdf_indexed(df, pdf_path):
    """
    Check if a PDF file has already been indexed in the database.
    Uses file hash for reliable checking.
    """
    if len(df) == 0:
        return False
    
    # Calculate hash of the PDF file
    with open(pdf_path, 'rb') as f:
        pdf_hash = hashlib.md5(f.read()).hexdigest()
    
    return pdf_hash in df['pdf_hash'].values if 'pdf_hash' in df.columns else False

# Load existing database
df_database = load_database(database_path)

## Step 5: Load Vision Embedding Model

In [ ]:
# Load the vision embedding model
print("Loading vision embedding model...")
processor = AutoImageProcessor.from_pretrained("nomic-ai/nomic-embed-vision-v1.5", use_fast=True)
vision_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-vision-v1.5", trust_remote_code=True)

print("Model loaded successfully")
print(f"Processor: {type(processor)}")
print(f"Vision model: {type(vision_model)}")

## Step 6: Process PDFs and Generate Embeddings

In [ ]:
def process_pdf_to_embeddings(pdf_path, pptx_name, processor, vision_model, images_dir, target_size=(854, 480)):
    """
    Convert PDF to images, resize them, and generate embeddings.
    Returns a list of dictionaries with image info and embeddings.
    """
    results = []
    
    # Calculate PDF hash
    with open(pdf_path, 'rb') as f:
        pdf_hash = hashlib.md5(f.read()).hexdigest()
    
    try:
        # Convert PDF to images
        print(f"Converting {pdf_path.name} to images...")
        images = convert_from_path(pdf_path)
        
        for slide_num, image in enumerate(images, start=1):
            try:
                # Resize image to target size
                image_resized = image.resize(target_size, Image.LANCZOS)
                
                # Convert to RGB if necessary
                if image_resized.mode != 'RGB':
                    image_resized = image_resized.convert('RGB')
                
                # Generate filename
                filename = f"{pptx_name}_slide_{slide_num:03d}.png"
                image_path = images_dir / filename
                
                # Save image
                image_resized.save(image_path)
                
                # Generate embedding
                inputs = processor(image_resized, return_tensors="pt")
                img_emb = vision_model(**inputs).last_hidden_state
                img_embeddings = F.normalize(img_emb[:, 0], p=2, dim=1)
                np_emb = img_embeddings.detach().numpy()[0]
                
                # Store result
                results.append({
                    'filename': filename,
                    'source_pptx': pptx_name,
                    'pdf_file': pdf_path.name,
                    'slide_number': slide_num,
                    'pdf_hash': pdf_hash,
                    'embedding': np_emb
                })
                
            except Exception as e:
                print(f"Error processing slide {slide_num} of {pdf_path.name}: {e}")
        
        print(f"Processed {len(results)} slides from {pdf_path.name}")
        
    except Exception as e:
        print(f"Error processing PDF {pdf_path.name}: {e}")
    
    return results

# Process all PDF/PPTX pairs
all_results = []
processed_count = 0
skipped_count = 0

for pdf_path, pptx_name in pdf_pptx_pairs:
    # Check if PDF is already indexed
    if is_pdf_indexed(df_database, pdf_path):
        print(f"Skipping {pdf_path.name} - already indexed")
        skipped_count += 1
        continue
    
    # Process the PDF
    results = process_pdf_to_embeddings(pdf_path, pptx_name, processor, vision_model, images_dir)
    all_results.extend(results)
    processed_count += 1

print(f"\nProcessing complete:")
print(f"  Processed: {processed_count} PDFs")
print(f"  Skipped: {skipped_count} PDFs (already indexed)")
print(f"  Total slides: {len(all_results)}")

## Step 7: Create DataFrame and Add UMAP Coordinates

In [ ]:
# Create DataFrame from new results
if all_results:
    df_new = pd.DataFrame(all_results)
    
    # Combine with existing database
    df_combined = pd.concat([df_database, df_new], ignore_index=True)
    
    print(f"Combined database now has {len(df_combined)} entries")
else:
    df_combined = df_database
    print("No new slides to process")

# Display sample
if len(df_combined) > 0:
    display(df_combined[['filename', 'source_pptx', 'slide_number']].head(10))

## Step 8: Generate UMAP Coordinates for All Embeddings

In [ ]:
if len(df_combined) > 0:
    from umap import UMAP
    
    # Convert embeddings list to numpy array matrix
    embedding_matrix = np.stack(df_combined['embedding'].values)
    
    print(f"Embedding matrix shape: {embedding_matrix.shape}")
    print("Creating 3D UMAP coordinates...")
    
    # Create 3D UMAP
    umap_reducer = UMAP(n_components=3, random_state=42)
    umap_coords = umap_reducer.fit_transform(embedding_matrix)
    
    # Add UMAP coordinates to dataframe
    df_combined['x'] = umap_coords[:, 0]
    df_combined['y'] = umap_coords[:, 1]
    df_combined['z'] = umap_coords[:, 2]
    
    print("UMAP coordinates created successfully")
    print(f"X range: {df_combined['x'].min():.2f} to {df_combined['x'].max():.2f}")
    print(f"Y range: {df_combined['y'].min():.2f} to {df_combined['y'].max():.2f}")
    print(f"Z range: {df_combined['z'].min():.2f} to {df_combined['z'].max():.2f}")
    
    display(df_combined[['filename', 'source_pptx', 'x', 'y', 'z']].head(10))
else:
    print("No data to process for UMAP")

## Step 9: Save Database to CSV

In [ ]:
if len(df_combined) > 0:
    # Convert embedding arrays to strings for CSV storage
    df_to_save = df_combined.copy()
    df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))
    
    # Save to CSV
    df_to_save.to_csv(database_path, index=False)
    
    print(f"Database saved to {database_path}")
    print(f"Total entries: {len(df_to_save)}")
    print(f"Columns: {list(df_to_save.columns)}")
else:
    print("No data to save")

## Step 10: Query Database Examples

In [ ]:
# Example queries on the database
if len(df_combined) > 0:
    print("Database Statistics:")
    print(f"Total slides: {len(df_combined)}")
    print(f"Unique presentations: {df_combined['source_pptx'].nunique()}")
    print(f"Unique PDF files: {df_combined['pdf_file'].nunique()}")
    
    print("\nSlides per presentation:")
    print(df_combined['source_pptx'].value_counts())
    
    # Example: Check if a specific PDF is indexed
    if len(pdf_pptx_pairs) > 0:
        test_pdf = pdf_pptx_pairs[0][0]
        is_indexed = is_pdf_indexed(df_combined, test_pdf)
        print(f"\nIs {test_pdf.name} indexed? {is_indexed}")